# What egocentric video embeddings encode, and how to combine them with inertial sensors

This notebook accompanies our paper and shows its central idea on this challenge's data, without temporal order.

- Paper: https://arxiv.org/abs/2610.12926
- Code that reproduces every table and figure: https://github.com/habakan/what-egocentric-embeddings-encode

In [ ]:
import os
from pathlib import Path
from IPython.display import Image, display

DATA = Path(os.environ.get("WEAR_NB_DATA", "/kaggle/input/wear-video-graph-inputs"))
if not (DATA / "train_tiles.npz").exists():                     # the mount path differs between Kaggle images
    DATA = next(Path("/kaggle/input").rglob("train_tiles.npz")).parent
display(Image(filename=str(DATA / "fig_teaser.png"), width=960))

*(a) The leading principal components of a WEAR recording's video embedding follow where the camera looks (sky, horizon,
close ground, the wearer's own limbs); the arrow is the direction in which the head tilt estimated from the frames increases.
(b) The video links a clip to its nearest clips, which come from the same set but carry other limbs; each clip is predicted from
its own accelerometer, and averaging over the links corrects it.*
*Frames from WEAR (Bock et al., 2024), CC BY-NC-SA 4.0, downscaled. This figure is shared under CC BY-NC-SA 4.0.*

**The idea.** Each test clip carries the accelerometer of one random limb, so a single-clip inertial classifier is weak.
But an exercise set lasts about half a minute, so many clips come from the same set, spread over the four limbs.
Clips of one set share the place, posture and viewing direction, so the egocentric video can find them.
We therefore let **the video link the clips** (a k-NN graph per participant) and **the inertial classifier label them**,
and propagate the inertial predictions over the video graph.

This notebook does not train the per-clip classifiers. It loads their out-of-fold and test probabilities from the attached dataset
(made with the code on GitHub) and runs everything after them: the video graph, the propagation, and the decision rule.
It reproduces the main table of the paper in subject-wise cross-validation and writes our final submission (public LB 0.894).
It does **not** use the adjacency of the shuffled test clips.

In [ ]:
display(Image(filename=str(DATA / "fig_setting.png"), width=720))

*The setting on 240 s of one training participant. (a) What was recorded: four limb sensors and sets lasting tens of seconds.
(b) What the test set keeps: one random limb per second, shuffled. A set still leaves many clips with similar video.*

In [ ]:
import time
import numpy as np
import torch
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt

DEV = "cuda" if torch.cuda.is_available() else "cpu"
C = 19
tr = dict(np.load(DATA / "train_tiles.npz"))
te = dict(np.load(DATA / "test.npz"))
print(DEV, {k: v.shape for k, v in tr.items()})

## The building blocks

Everything below runs per participant, on that participant's clips only.

- `knn_graph`: mutual k-nearest neighbours by cosine similarity.
- `propagate`: label propagation, `Q ← (1-α) P + α · (weighted mean of the neighbours' Q)`, anchored to the classifier's `P`.
- Between stages, the graph is rebuilt from `[video features ‖ λ√Q]`, so the refined labels sharpen the neighbourhoods.
- After every stage, the video-only probabilities `V` are pooled log-linearly (total weight `g`).
- `drop_top_pcs`: remove the top `m` principal components of the participant's video features before building the graph.

In [ ]:
display(Image(filename=str(DATA / "fig_pipeline.png"), width=900))

In [ ]:
def _t(a):
    return torch.as_tensor(np.asarray(a), dtype=torch.float64, device=DEV)


def knn_graph(F, k, mutual=True):
    n = F.shape[0]
    F = F / (torch.linalg.norm(F, dim=1, keepdim=True) + 1e-8)
    S = F @ F.T
    S.fill_diagonal_(-float("inf"))
    k = min(k, n - 2)
    w, nn = torch.topk(S, k, dim=1)
    if mutual:
        r = torch.zeros((n, n), dtype=torch.bool, device=F.device)
        r.scatter_(1, nn, True)
        keep = r[nn, torch.arange(n, device=F.device)[:, None].expand_as(nn)]
        w = torch.where(keep, w, torch.full_like(w, -1e9))
    return nn, w


def propagate(P, nn, w, alpha, iters, sharp=10.0):
    ww = torch.exp((w - w.max(1, keepdim=True).values) * sharp)
    ww = ww / (ww.sum(1, keepdim=True) + 1e-12)
    Q = P.clone()
    for _ in range(iters):
        Q = (1 - alpha) * P + alpha * (Q[nn] * ww[:, :, None]).sum(1)
    return Q


def sharpen(Q, temp):
    R = torch.clamp(Q, min=1e-9) ** temp
    return R / R.sum(1, keepdim=True)


def gate_blend(Q, V, g):
    if g == 0:
        return Q
    lg = (1 - g) * torch.log(torch.clamp(Q, min=1e-9)) + g * torch.log(torch.clamp(V, min=1e-9))
    R = torch.exp(lg - lg.max(1, keepdim=True).values)
    return R / R.sum(1, keepdim=True)


def drop_top_pcs(F, m):
    if m <= 0:
        return F
    c = F - F.mean(0)
    _, _, Vh = torch.linalg.svd(c, full_matrices=False)
    B = Vh[:min(m, Vh.shape[0])]
    return c - (c @ B.T) @ B

### The protocol as a constraint (final submission only)

WEAR has a fixed protocol: every participant performs all eighteen activities for roughly the same time.
In the training recordings an activity occupies 78–126 one-second clips (5–95% range), and in the three pairs of exercise variants
the complex variant takes 45–58% of the pair. The final submission nudges the class scores after every stage until each
recording's predicted counts fall inside these ranges (`β = 0.5`); `NULL` takes the rest. This uses no test labels,
but it assumes that the test participants followed the protocol.

In [ ]:
LO, HI = 78, 126
PAIRS = [(11, 12), (13, 14), (16, 17)]      # (plain, complex)


def band_bias(L, lo=LO, hi=HI, step=0.1, iters=200):
    b = np.zeros(L.shape[1])
    for _ in range(iters):
        cnt = np.bincount((L + b).argmax(1), minlength=L.shape[1])
        up, dn = cnt[1:] < lo, cnt[1:] > hi
        if not (up.any() or dn.any()):
            break
        b[1:] += step * up - step * dn
    return b


def pair_bias(L, lo, hi, step=0.05, iters=200):
    b = np.zeros(L.shape[1])
    for a, c in PAIRS:
        d = 0.0
        for _ in range(iters):
            p = (L + b).argmax(1)
            na, nc = (p == a).sum(), (p == c).sum()
            if na + nc == 0:
                break
            sh = nc / (na + nc)
            if lo <= sh <= hi:
                break
            d += step if sh < lo else -step
            b[a], b[c] = -d, d
    return b


def apply_band(Q, rec, beta, pair=None):
    # per recording (a training recording, or one test participant)
    if beta == 0:
        return Q
    L = torch.log(torch.clamp(Q, min=1e-9)).cpu().numpy()
    B = np.zeros_like(L)
    for r in np.unique(rec):
        i = rec == r
        B[i] = beta * band_bias(L[i])
        if pair is not None:
            B[i] += pair[2] * pair_bias(L[i] + B[i], pair[0], pair[1])
    R = Q * torch.exp(_t(B))
    return R / R.sum(1, keepdim=True)

In [ ]:
def system(P, G, V, rec, stages=(4.0, 4.0), g=0.2, beta=0.0, pair=None, k=30, alpha=0.75, iters=5, temp=1.4):
    # one participant. G: graph features, rec: recording of every clip (the band acts per recording)
    gi = g / (len(stages) + 1)
    band = lambda Q: apply_band(Q, rec, beta, pair)
    Fn = G - G.mean(0)
    Fn = Fn / (torch.linalg.norm(Fn, dim=1, keepdim=True) + 1e-8)
    nn, w = knn_graph(Fn, k)
    Q = band(gate_blend(sharpen(propagate(P, nn, w, alpha, iters), temp), V, gi))
    for lam in stages:
        nn, w = knn_graph(torch.cat([Fn, lam * torch.sqrt(Q)], 1), k)
        Q = band(gate_blend(sharpen(propagate(Q, nn, w, alpha, iters), temp), V, gi))
    return Q


def graph_features(F, U, m):
    f = drop_top_pcs(_t(F), m)
    f = f - f.mean(0)
    f = f / (torch.linalg.norm(f, dim=1, keepdim=True) + 1e-8)
    return torch.cat([f, 0.5 * torch.sqrt(_t(U))], 1)


def run_all(d, m=30, **kw):
    Q = np.empty_like(d["P"])
    for s in np.unique(d["sbj"]):
        i = d["sbj"] == s
        Q[i] = system(_t(d["P"][i]), graph_features(d["F"][i], d["U"][i], m), _t(d["V"][i]), d["rec"][i], **kw).cpu().numpy()
    return Q


def macro_f1(y, p):
    return f1_score(y, p, labels=list(range(C)), average="macro", zero_division=0)


def prior_correct(Q, tau):
    Q = Q / Q.sum(1, keepdims=True)
    return Q / np.power(Q.mean(0) + 1e-9, tau)          # prior averaged over all clips being labelled


def tune_tau(Q, y, taus=np.linspace(0, 2.0, 41)):
    return max((macro_f1(y, prior_correct(Q, t).argmax(1)), t) for t in taus)

## Test-like validation data

The training recordings are cut like the test set: non-overlapping one-second tiles, one random valid limb per tile.
The limb assignment is random, so we repeat it over seeds. All probabilities are out-of-fold (five subject-wise folds).

In [ ]:
def eval_set(seed):
    rng = np.random.RandomState(seed)
    n = len(tr["label"])
    sens = rng.randint(0, 4, n)
    idx = np.where(tr["valid"][np.arange(n), sens])[0]
    sens = sens[idx]
    P = tr["P"][idx, sens]
    keep = P.sum(1) > 1e-6
    idx, sens, P = idx[keep], sens[keep], P[keep]
    return dict(P=P / P.sum(1, keepdims=True), U=tr["U"][idx, sens], V=tr["V"][idx, sens], F=tr["F"][idx],
                y=tr["label"][idx], sbj=tr["sbj"][idx], rec=tr["rec"][idx])


d = eval_set(42)
print(len(d["y"]), "clips from", len(np.unique(d["sbj"])), "participants")

## The main table of the paper

| row | what changes |
|---|---|
| inertial classifier | `argmax P / π^τ`, no graph |
| + propagation | video graph, three stages, `g = 0`, no component removal |
| + video pooling | `g = 0.2` |
| + component removal | top 30 principal components of the video features removed (paper system) |

Paper (three seeds): 0.648, 0.830, 0.843, 0.852.

In [ ]:
SEEDS = [42, 7, 123]
rows = {"inertial classifier": [], "+ propagation": [], "+ video pooling": [], "+ component removal": []}
t0 = time.time()
for seed in SEEDS:
    d = eval_set(seed)
    rows["inertial classifier"].append(tune_tau(d["P"], d["y"])[0])
    rows["+ propagation"].append(tune_tau(run_all(d, m=0, g=0.0), d["y"])[0])
    rows["+ video pooling"].append(tune_tau(run_all(d, m=0, g=0.2), d["y"])[0])
    rows["+ component removal"].append(tune_tau(run_all(d, m=30, g=0.2), d["y"])[0])
for k, v in rows.items():
    print(f"{k:22s} {np.mean(v):.4f}  ({' '.join(f'{x:.4f}' for x in v)})")
print(f"{time.time() - t0:.0f}s")

## Removing the leading components: purity falls, the score rises

The leading principal components of an egocentric embedding are the most activity-discriminative ones. Beyond the first few,
removing them lowers the label purity of the video neighbourhoods (0.73 at `m = 0`, 0.63 at `m = 30`).
Yet the final score rises up to about `m = 30`, so the usual measure of graph quality predicts the opposite of what the system does.
In the paper we show that on Ego-Exo4D these components follow the measured head orientation (for four encoders, not for exocentric cameras),
and on WEAR the scene layout. Truncated whitening of the same directions gives a similar gain,
so the gain may come from how much the leading directions weigh in the similarities rather than from removing their content.

In [ ]:
display(Image(filename=str(DATA / "fig_pcends.png"), width=960))

*Frames at the two ends of the first and second principal components for three WEAR participants: the components follow the view.
Frames from WEAR (Bock et al., 2024), CC BY-NC-SA 4.0, downscaled. This figure is shared under CC BY-NC-SA 4.0.*

In [ ]:
def purity(d, m, k=30):
    out = []
    for s in np.unique(d["sbj"]):
        i = d["sbj"] == s
        f = drop_top_pcs(_t(d["F"][i]), m)
        f = f - f.mean(0)
        nn, _ = knn_graph(f, k, mutual=False)
        y = _t(d["y"][i]).long()
        out.append((y[nn] == y[:, None]).double().mean().item())
    return float(np.mean(out))


d = eval_set(42)
MS = [0, 5, 10, 20, 30, 50, 80, 120]
f1s = [tune_tau(run_all(d, m=m), d["y"])[0] for m in MS]
pur = [purity(d, m) for m in MS]
for m, a, b in zip(MS, f1s, pur):
    print(f"m={m:3d}  macro-F1 {a:.4f}  neighbour purity {b:.3f}")

fig, ax = plt.subplots(figsize=(6.4, 3.6))
ax.plot(MS, f1s, "o-", color="#2a78d6", label="macro-F1 of the system")
ax.set_xlabel("top principal components removed (m)"); ax.set_ylabel("macro-F1", color="#2a78d6")
ax2 = ax.twinx()
ax2.plot(MS, pur, "s--", color="#eb6834", label="neighbour label purity")
ax2.set_ylabel("neighbour label purity", color="#eb6834")
ax.set_title("Removing leading components: purity falls, the score rises (seed 42)")
fig.tight_layout(); plt.show()

## Final submission

Our final submission adds the protocol constraint between the stages (`β = 0.5`, with the variant-pair range) and uses `λ = 2`.
Cross-validation: 0.886 (three seeds), public LB 0.894. `τ` is chosen out-of-fold on seed 42 and reused for the test set.
Each test participant has one recording, so the band acts per participant.

In [ ]:
FINAL = dict(stages=(2.0, 2.0), beta=0.5, pair=(0.45, 0.58, 0.5))
d = eval_set(42)
f1, tau = tune_tau(run_all(d, **FINAL), d["y"])
print(f"out-of-fold macro-F1 (seed 42) {f1:.4f}, tau {tau:.2f}")

t = dict(P=te["P"], U=te["U"], V=te["V"], F=te["F"], sbj=te["sbj"], rec=te["sbj"].astype(str))
Qt = run_all(t, **FINAL)
pred = prior_correct(Qt, tau).argmax(1)
import pandas as pd
pd.DataFrame({"id": te["id"], "target_feature": pred}).to_csv("submission.csv", index=False)
print("NULL share", round((pred == 0).mean(), 3), "| clips per class", np.bincount(pred, minlength=C)[1:].min(), "-", np.bincount(pred, minlength=C)[1:].max())

## More in the paper

- On Ego-Exo4D, measured head orientation explains the leading components of the egocentric embedding (R² 0.49–0.70) but not of the exocentric cameras.
- Splitting the WEAR embedding into what the four limbs predict and the rest: the limb-predicted part separates activities across people,
  the rest separates exercise sets within a participant.
- With the same inertial classifier, propagation over an inertial graph lowers macro-F1 (−0.060) and over the video graph raises it (+0.167).
  Every fusion variant we tried did worse than this division of roles.
- Notes on the benchmark: label errors in the training data, and the adjacency of shuffled clips.